# Two Pointers

The **two pointer** technique uses two indexes that walk through a sequence together, so a problem that looks like it needs nested loops (`O(n²)`) can be solved in a single pass (`O(n)`).

There are three common shapes:

| Shape | How the pointers move | Typical use |
|---|---|---|
| **Opposite ends** | `left` starts at 0, `right` at the end, they move toward each other | sorted data, pairs, palindromes |
| **Same direction (slow / fast)** | `slow` marks where to write, `fast` scans ahead | in-place filtering, removing duplicates |
| **Two sequences** | one pointer per list, advance whichever is "behind" | merging, intersecting sorted lists |

Contents (each with a real-world story):
1. Gift card shopping: pair that sums to a target (opposite ends)
2. In-flight movies: closest pair under a limit (opposite ends)
3. Rescue boats / ride sharing: minimum trips (opposite ends, greedy)
4. Username / plate validation: palindrome check (opposite ends)
5. Cleaning sensor logs: remove duplicates in place (slow / fast)
6. Inventory: push out-of-stock items to the end (slow / fast)
7. Support tickets: sort by priority in one pass (three pointers, Dutch flag)
8. Bank statements: merge two sorted transaction feeds (two sequences)
9. Meeting scheduler: common free time of two people (two sequences)
10. Practice problems

In [ ]:
nums = [num for num in range(1, 11, 1) if num % 2 == 0]
print(nums)

[2, 4, 6, 8, 10]


In [15]:
for num in nums:
    print(num)

2
4
6
8
10


In [14]:
for i in range(len(nums)):
    print(nums[i])

2
4
6
8
10


In [19]:
index = 0
length = len(nums)
while index < length:
    print(nums[index])
    index = index + 1

2
4
6
8
10


In [45]:
nums.append(12)
cond = True
pl = 0
length = len(nums) - 1
pr = length
while cond:
    print(f"num left : {nums[pl]}, pl : {pl}")
    print(f"num right : {nums[pr]}, pr : {pr}")

    if pr <= pl:
        cond = False
    pr -= 1 
    pl += 1


num left : 2, pl : 0
num right : 12, pr : 6
num left : 4, pl : 1
num right : 12, pr : 5
num left : 6, pl : 2
num right : 10, pr : 4
num left : 8, pl : 3
num right : 8, pr : 3


## 1. Gift card shopping: find two items that use the card exactly

**Story:** You have a $100 gift card and want to buy exactly **two** items that use the whole balance. The store's price list is sorted.

**Brute force:** try every pair: `O(n²)`.

**Two pointers:** start with the cheapest (`left`) and most expensive (`right`).
- sum too small → need more money → move `left` right
- sum too big → need less money → move `right` left
- equal → found it

Each step discards one item forever, so it's `O(n)`. **This only works because the list is sorted.**

In [ ]:
def find_gift_pair(prices, card_balance):
    """Return (i, j) of two items whose prices add up to card_balance, or None."""
    left, right = 0, len(prices) - 1

    while left < right:
        total = prices[left] + prices[right]
        if total == card_balance:
            return left, right
        elif total < card_balance:
            left += 1      # need a bigger sum
        else:
            right -= 1     # need a smaller sum
    return None


prices = [5, 12, 18, 25, 40, 55, 70, 88]   # sorted price list
pair = find_gift_pair(prices, 100)
print(pair)                                # (1, 7)
print(prices[pair[0]], prices[pair[1]])    # 12 88

Let's trace it to see *why* no pair is ever missed:

```
prices = [5, 12, 18, 25, 40, 55, 70, 88], target = 100
 L=5   R=88  -> 93  < 100  move L
 L=12  R=88  -> 100 == 100 found!
```

The first match found is `12 + 88`. (If several pairs work, this returns the first one it meets. `25 + 70 = 95` would be found for a $95 card, as the trace below shows.)

In [ ]:
# trace version so you can watch the pointers move
def find_gift_pair_trace(prices, card_balance):
    left, right = 0, len(prices) - 1
    while left < right:
        total = prices[left] + prices[right]
        print(f"left={prices[left]:>3}  right={prices[right]:>3}  sum={total}")
        if total == card_balance:
            return prices[left], prices[right]
        elif total < card_balance:
            left += 1
        else:
            right -= 1
    return None

print(find_gift_pair_trace([5, 12, 18, 25, 40, 55, 70, 88], 95))
print(find_gift_pair_trace([5, 12, 18, 25], 1000))   # no pair -> None

## 2. In-flight entertainment: two movies that fill the flight

**Story:** (a classic Amazon interview question) A flight lasts `D` minutes. You must watch exactly **two** movies, finishing **at least 30 minutes before landing**. Pick the pair with the **longest total runtime** that still fits.

Same opposite-ends idea, but instead of "exact match" we track the **best so far**:
- sum fits → record it, then try a bigger sum (`left += 1`)
- sum too big → shrink (`right -= 1`)

We sort first (`O(n log n)`), keeping the original indexes so we can report which movies.

In [ ]:
def best_movie_pair(durations, flight_minutes):
    limit = flight_minutes - 30
    # sort by duration but remember original positions
    movies = sorted(enumerate(durations), key=lambda m: m[1])

    left, right = 0, len(movies) - 1
    best_total, best_pair = -1, None

    while left < right:
        total = movies[left][1] + movies[right][1]
        if total <= limit:
            if total > best_total:
                best_total = total
                best_pair = (movies[left][0], movies[right][0])
            left += 1
        else:
            right -= 1
    return best_pair, best_total


durations = [90, 85, 75, 60, 120, 150, 125]
pair, total = best_movie_pair(durations, 250)    # must fit in 220 min
print(pair, total)
print([durations[i] for i in pair])

## 3. Rescue boats / ride sharing: minimum number of trips

**Story:** A rescue boat (or a 2-seat bike taxi) carries **at most 2 people** and has a weight limit. Given everyone's weight, what's the minimum number of trips?

**Greedy + two pointers:** sort weights. The heaviest person must go *some* time. Try pairing them with the lightest:
- if they fit together → both go, move both pointers
- otherwise → heaviest goes alone, move `right` only

Pairing the heaviest with the lightest is optimal: if the lightest can't ride with them, nobody can.

In [ ]:
def min_trips(weights, limit):
    weights = sorted(weights)
    left, right = 0, len(weights) - 1
    trips = 0

    while left <= right:            # <= so a single remaining person still gets a trip
        if weights[left] + weights[right] <= limit:
            left += 1               # lightest rides along
        right -= 1                  # heaviest always rides
        trips += 1
    return trips


print(min_trips([70, 50, 80, 50], 100))       # 3 -> (50,50) (70) (80)
print(min_trips([30, 40, 60, 70, 90], 100))   # 3 -> (30,70) (40,60) (90)
print(min_trips([100, 100, 100], 100))        # 3 -> everyone alone

## 4. Validation: is it a palindrome?

**Story:** A vanity license plate site, a puzzle game, or a DNA analysis tool (palindromic sequences are where restriction enzymes cut DNA) needs to check whether text reads the same both ways, **ignoring spaces, punctuation and case**.

`s == s[::-1]` works but builds a cleaned copy and a reversed copy. Two pointers check in place, skip junk characters, and stop at the first mismatch.

In [ ]:
def is_palindrome(text):
    left, right = 0, len(text) - 1

    while left < right:
        if not text[left].isalnum():       # skip spaces / punctuation
            left += 1
        elif not text[right].isalnum():
            right -= 1
        elif text[left].lower() != text[right].lower():
            return False                   # early exit on first mismatch
        else:
            left += 1
            right -= 1
    return True


for s in ["A man, a plan, a canal: Panama", "RACECAR", "Was it a car or a cat I saw?", "python"]:
    print(f"{s!r:35} -> {is_palindrome(s)}")

In [ ]:
# DNA version: a site is a "reverse-complement palindrome" if the strand
# equals its complement read backwards (A<->T, C<->G). Example: EcoRI cuts GAATTC.
COMPLEMENT = {"A": "T", "T": "A", "C": "G", "G": "C"}

def is_restriction_site(dna):
    left, right = 0, len(dna) - 1
    while left <= right:
        if dna[left] != COMPLEMENT[dna[right]]:
            return False
        left += 1
        right -= 1
    return True

print(is_restriction_site("GAATTC"))   # True  (EcoRI)
print(is_restriction_site("GGATCC"))   # True  (BamHI)
print(is_restriction_site("GATTAC"))   # False

## 5. Cleaning sensor logs: remove consecutive duplicates in place

**Story:** A temperature sensor logs a reading every second, but the value rarely changes. Before uploading, you want to keep only the readings where the value **changed**, editing the list **in place** (on a tiny IoT device memory is tight, so no second list).

**Slow / fast pointers:**
- `fast` scans every reading
- `slow` points at the last *kept* reading
- when `fast` sees something new, copy it to `slow + 1`

In [ ]:
def compress_readings(readings):
    """Remove consecutive duplicates in place. Returns the new length."""
    if not readings:
        return 0

    slow = 0
    for fast in range(1, len(readings)):
        if readings[fast] != readings[slow]:
            slow += 1
            readings[slow] = readings[fast]
    del readings[slow + 1:]        # trim the leftovers
    return slow + 1


temps = [21, 21, 21, 22, 22, 23, 23, 23, 23, 22, 22, 21]
n = compress_readings(temps)
print(n, temps)    # 5 [21, 22, 23, 22, 21]

## 6. Inventory: move out-of-stock items to the end

**Story:** An online store shows products in a ranked order. Products with `0` stock should drop to the bottom, but **in-stock items must keep their ranking order**.

Same slow / fast shape: `slow` is where the next in-stock item goes; swapping pushes the zero-stock ones backward.

In [ ]:
def push_out_of_stock_to_end(products):
    """products: list of (name, stock). Reorders in place, stable for in-stock items."""
    slow = 0
    for fast in range(len(products)):
        if products[fast][1] > 0:
            products[slow], products[fast] = products[fast], products[slow]
            slow += 1
    return products


catalog = [("Laptop", 5), ("Mouse", 0), ("Monitor", 2), ("Webcam", 0), ("Keyboard", 7), ("Hub", 1)]
for name, stock in push_out_of_stock_to_end(catalog):
    print(f"{name:10} {stock}")

## 7. Support tickets: sort by priority in one pass (Dutch National Flag)

**Story:** A help desk queue has tickets tagged `HIGH`, `MEDIUM`, `LOW`. Arrange them HIGH → MEDIUM → LOW **in one pass, in place**.

With only 3 categories we don't need a general sort. Use **three pointers**:
- `low`: next slot for HIGH (front region)
- `high`: next slot for LOW (back region)
- `mid`: the scanner

```
[ HIGH... | MEDIUM... | unknown... | LOW... ]
          low         mid          high
```

In [ ]:
RANK = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}

def sort_tickets(tickets):
    low, mid, high = 0, 0, len(tickets) - 1

    while mid <= high:
        rank = RANK[tickets[mid][1]]
        if rank == 0:          # HIGH -> swap to front
            tickets[low], tickets[mid] = tickets[mid], tickets[low]
            low += 1
            mid += 1
        elif rank == 1:        # MEDIUM -> already in the middle
            mid += 1
        else:                  # LOW -> swap to back; don't move mid, the swapped-in item is unchecked
            tickets[mid], tickets[high] = tickets[high], tickets[mid]
            high -= 1
    return tickets


queue = [("T1", "LOW"), ("T2", "HIGH"), ("T3", "MEDIUM"), ("T4", "HIGH"),
         ("T5", "LOW"), ("T6", "MEDIUM"), ("T7", "HIGH")]
for ticket in sort_tickets(queue):
    print(ticket)

Note: this is **not stable**: `T7` jumped ahead of `T2` even though both are HIGH. If ticket age matters within a priority, use `sorted(queue, key=lambda t: RANK[t[1]])` instead (stable, `O(n log n)`).

## 8. Bank statements: merge two sorted transaction feeds

**Story:** Your budgeting app pulls transactions from a checking account and a credit card. Each feed is already sorted by date. Show one combined timeline.

One pointer per list. Compare the two "heads", take the earlier one, advance that pointer. This is exactly the **merge** step of merge sort: `O(n + m)`.

In [ ]:
def merge_feeds(checking, credit):
    i, j = 0, 0
    merged = []

    while i < len(checking) and j < len(credit):
        if checking[i][0] <= credit[j][0]:    # compare dates (ISO strings sort correctly)
            merged.append(checking[i])
            i += 1
        else:
            merged.append(credit[j])
            j += 1

    # one list is exhausted; the rest of the other is already sorted
    merged.extend(checking[i:])
    merged.extend(credit[j:])
    return merged


checking = [("2026-10-01", "Salary", +3000), ("2026-10-03", "Rent", -1200), ("2026-10-09", "ATM", -100)]
credit   = [("2026-10-02", "Groceries", -85), ("2026-10-03", "Gas", -40), ("2026-10-05", "Netflix", -15),
            ("2026-10-12", "Flight", -320)]

for date, desc, amount in merge_feeds(checking, credit):
    print(f"{date}  {desc:10} {amount:>6}")

## 9. Meeting scheduler: when are both people free?

**Story:** Calendar apps (Google Calendar's "Find a time", Calendly) intersect two people's free slots. Each person's free slots are sorted and non-overlapping.

Two pointers, one per calendar:
- overlap of two slots = `(max(starts), min(ends))`, valid if start < end
- advance whichever slot **ends first**: it can't overlap anything later in the other list

In [ ]:
def common_free_time(alice, bob, min_minutes=0):
    """Slots are (start, end) in minutes since midnight."""
    i, j = 0, 0
    result = []

    while i < len(alice) and j < len(bob):
        start = max(alice[i][0], bob[j][0])
        end = min(alice[i][1], bob[j][1])
        if end - start >= max(min_minutes, 1):
            result.append((start, end))

        if alice[i][1] < bob[j][1]:   # whoever finishes first moves on
            i += 1
        else:
            j += 1
    return result


def hm(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

alice = [(9*60, 10*60+30), (12*60, 13*60), (15*60, 17*60)]
bob   = [(10*60, 12*60+30), (14*60+30, 15*60+45), (16*60+30, 18*60)]

for start, end in common_free_time(alice, bob):
    print(hm(start), "-", hm(end))

print("at least 45 min:", [(hm(s), hm(e)) for s, e in common_free_time(alice, bob, 45)])

## Quick check: all examples

Run this cell to verify every function with `assert`s. No output means everything passed.

In [ ]:
assert find_gift_pair([5, 12, 18, 25, 40, 55, 70, 88], 100) == (1, 7)
assert find_gift_pair([1, 2, 3], 100) is None
assert best_movie_pair([90, 85, 75, 60, 120, 150, 125], 250)[1] == 215
assert min_trips([3, 2, 2, 1], 3) == 3
assert is_palindrome("No 'x' in Nixon") and not is_palindrome("hello")
data = [1, 1, 2, 2, 2, 3]; assert compress_readings(data) == 3 and data == [1, 2, 3]
assert [p[0] for p in push_out_of_stock_to_end([("a", 0), ("b", 1), ("c", 0), ("d", 2)])] [:2] == ["b", "d"]
assert [t[1] for t in sort_tickets([("x", "LOW"), ("y", "HIGH"), ("z", "MEDIUM")])] == ["HIGH", "MEDIUM", "LOW"]
assert len(merge_feeds(checking, credit)) == 7
assert common_free_time([(0, 10)], [(5, 15)]) == [(5, 10)]
print("all good")

## 10. Practice problems

Try these yourself; each has a real-world spin and uses one of the shapes above.

1. **Container with most water** (opposite ends): fence posts of different heights stand in a row. Pick two posts that hold the most water between them. *Hint: always move the shorter post inward.*
2. **Three-item bundle** (sort + two pointers inside a loop): find all triples of prices that sum exactly to a bundle price, no duplicates (LeetCode "3Sum").
3. **Squares of a sorted list** (opposite ends): temperature deltas `[-7, -3, 0, 2, 5]` sorted; return their squares sorted in `O(n)`. *Hint: the biggest square is at one of the two ends.*
4. **Subsequence check** (two sequences): did the user type the letters of a shortcut command in order? e.g. `"gco"` is a subsequence of `"git checkout"`.
5. **Linked list cycle** (slow / fast, Floyd's algorithm): detect if a playlist set to "next song" ever loops back. Slow moves 1 step, fast moves 2; if they meet, there's a loop.

### When to reach for two pointers
- The data is **sorted** (or you can afford to sort it) and you're looking for a **pair**
- You need to modify an array **in place** with `O(1)` extra memory
- You're walking **two sorted sequences** at once
- You want to compare things from **both ends** (palindromes, symmetry)